<a href="https://colab.research.google.com/github/charlien12/ML-Projects/blob/main/ImageClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [5]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
train_df=CIFAR10(root='./sample_data',train=True,download=True,transform=transform)
test_df=CIFAR10(root='./sample_data',train=False,download=True,transform=transform)

100%|██████████| 170M/170M [01:49<00:00, 1.55MB/s]


In [6]:
train_loader=DataLoader(train_df,batch_size=64,shuffle=True)
test_loader=DataLoader(test_df,batch_size=64,shuffle=False)


# Build The CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # kernel size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2) ,

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening
        x = self.fc_layers(x)

        return x

In [8]:
model = CNN()

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [10]:
epochs = 20

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)}")

epoch=1/20 & loss=1.385721410174504
epoch=2/20 & loss=0.9359734489027497
epoch=3/20 & loss=0.7460293100236932
epoch=4/20 & loss=0.6247268917463015
epoch=5/20 & loss=0.519358184293408
epoch=6/20 & loss=0.4285925829120914
epoch=7/20 & loss=0.3464670218336765
epoch=8/20 & loss=0.27164686661775767
epoch=9/20 & loss=0.2141001431361946
epoch=10/20 & loss=0.16268325884781226
epoch=11/20 & loss=0.13471732801421904
epoch=12/20 & loss=0.10975376073666432
epoch=13/20 & loss=0.10019792322143245
epoch=14/20 & loss=0.09010830821821947
epoch=15/20 & loss=0.08746926801676964
epoch=16/20 & loss=0.07686224640370883
epoch=17/20 & loss=0.0797786856337529
epoch=18/20 & loss=0.06903750119689743
epoch=19/20 & loss=0.06778623317481469
epoch=20/20 & loss=0.06354326785832663


In [11]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 75.33
